# MagiV2 Sequence Inspector

Give one sequence of pages, run MagiV2 once, then print only the outputs you want.

Views:
- `text_boxes`
- `char_boxes`
- `panel_boxes`
- `tail_boxes`
- `text_char`
- `text_tail`
- `same_char_page`
- `cross_page_candidates`
- `ocr`
- `all`

`same_char_page` uses MagiV2's page-local character clusters.

`cross_page_candidates` uses MagiV2 crop embeddings + cosine similarity across different pages. It is a candidate matcher, not a guaranteed identity label.

In [1]:
from pathlib import Path
from PIL import Image
import numpy as np
import torch
from transformers import AutoModel

MODEL_PATH = r"C:\MODELS\magiv2"

model = AutoModel.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    local_files_only=True,
).cuda().eval()

print(type(model))

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
c:\Users\ESHAAN\BYTE\Applied-Task\.venvAppliedTask\Lib\site-packages\torch\nn\modules\module.py:2593: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(
c:\Users\ESHAAN\BYTE\Applied-Task\.venvAppliedTask\Lib\site-packages\torch\nn\modules\batchnorm.py:141: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (D

<class 'transformers_modules.magiv2.modelling_magiv2.Magiv2Model'>


In [2]:
def read_image(path):
    with open(path, "rb") as f:
        return np.array(Image.open(f).convert("L").convert("RGB"))

## 1. Give a sequence

In [3]:
SEQUENCE_ROOT = Path(
    r"C:\Users\ESHAAN\BYTE\Applied-Task\Magi\TaskData\development\images\seq_0dc2e3d874c804fc"
)

PAGE_FILES = sorted(SEQUENCE_ROOT.glob("*.png"))
chapter_pages = [read_image(p) for p in PAGE_FILES]

for i, p in enumerate(PAGE_FILES, start=1):
    print(i, p.name)

1 01.png
2 02.png
3 03.png


## 2. Run MagiV2 once

In [4]:
DO_OCR = True

empty_character_bank = {
    "images": [],
    "names": [],
}

with torch.no_grad():
    per_page_results = model.do_chapter_wide_prediction(
        chapter_pages,
        empty_character_bank,
        use_tqdm=True,
        do_ocr=DO_OCR,
    )

print(per_page_results[0].keys())

100%|██████████| 1/1 [00:04<00:00,  4.56s/it]

dict_keys(['panels', 'texts', 'characters', 'tails', 'text_character_associations', 'text_tail_associations', 'character_cluster_labels', 'is_essential_text', 'ocr', 'character_names'])


## 3. Helpers

In [5]:
def fmt_box(box):
    return [round(float(x), 1) for x in box]

def page_header(page_idx):
    return f"\n===== PAGE {page_idx + 1}: {PAGE_FILES[page_idx].name} ====="

def char_ids(result):
    return [f"C{i+1}" for i in range(len(result["characters"]))]

def text_ids(result):
    return [f"T{i+1}" for i in range(len(result["texts"]))]

def tail_ids(result):
    return [f"TAIL{i+1}" for i in range(len(result["tails"]))]

def panel_ids(result):
    return [f"PN{i+1}" for i in range(len(result["panels"]))]

## 4. Selective outputs

In [6]:
def show_text_boxes(results):
    for p, r in enumerate(results):
        print(page_header(p))
        for tid, box in zip(text_ids(r), r["texts"]):
            print(f"{tid}: {fmt_box(box)}")

def show_char_boxes(results):
    for p, r in enumerate(results):
        print(page_header(p))
        for cid, box in zip(char_ids(r), r["characters"]):
            print(f"{cid}: {fmt_box(box)}")

def show_panel_boxes(results):
    for p, r in enumerate(results):
        print(page_header(p))
        for pid, box in zip(panel_ids(r), r["panels"]):
            print(f"{pid}: {fmt_box(box)}")

def show_tail_boxes(results):
    for p, r in enumerate(results):
        print(page_header(p))
        for tid, box in zip(tail_ids(r), r["tails"]):
            print(f"{tid}: {fmt_box(box)}")

In [7]:
def show_text_char(results):
    for p, r in enumerate(results):
        print(page_header(p))
        tids = text_ids(r)
        cids = char_ids(r)

        for text_idx, char_idx in r["text_character_associations"]:
            line = f"{tids[text_idx]} -> {cids[char_idx]}"
            if "ocr" in r and text_idx < len(r["ocr"]):
                line += f" | {r['ocr'][text_idx]}"
            print(line)

def show_text_tail(results):
    for p, r in enumerate(results):
        print(page_header(p))
        tids = text_ids(r)
        tails = tail_ids(r)

        for text_idx, tail_idx in r["text_tail_associations"]:
            print(f"{tids[text_idx]} -> {tails[tail_idx]}")

## 5. Same character within a page

In [8]:
def show_same_char_page(results):
    for p, r in enumerate(results):
        print(page_header(p))

        groups = {}
        for cid, cluster_id in zip(char_ids(r), r["character_cluster_labels"]):
            groups.setdefault(int(cluster_id), []).append(cid)

        for cluster_id, members in groups.items():
            print(f"P{cluster_id + 1}: {members}")

## 6. OCR only

In [9]:
def show_ocr(results):
    for p, r in enumerate(results):
        print(page_header(p))

        if "ocr" not in r:
            print("OCR was not run.")
            continue

        for tid, text in zip(text_ids(r), r["ocr"]):
            print(f"{tid}: {text}")

## 7. Cross-page character candidates

This uses MagiV2's ViT-MAE crop embeddings. It compares every detected character on one page against every detected character on later pages.

In [10]:
@torch.no_grad()
def get_character_embeddings(pages, results):
    boxes = [r["characters"] for r in results]
    per_page = model.predict_crop_embeddings(pages, boxes)

    normalized = []
    for emb in per_page:
        if len(emb) == 0:
            normalized.append(emb)
        else:
            normalized.append(
                torch.nn.functional.normalize(emb, p=2, dim=1)
            )
    return normalized

character_embeddings = get_character_embeddings(
    chapter_pages,
    per_page_results,
)

In [11]:
def show_cross_page_candidates(results, embeddings, threshold=0.70):
    for pa in range(len(results)):
        for pb in range(pa + 1, len(results)):
            ea = embeddings[pa]
            eb = embeddings[pb]

            if len(ea) == 0 or len(eb) == 0:
                continue

            sims = ea @ eb.T
            print(f"\n===== PAGE {pa+1} ↔ PAGE {pb+1} =====")

            found = False
            for i in range(sims.shape[0]):
                for j in range(sims.shape[1]):
                    score = float(sims[i, j])

                    if score >= threshold:
                        print(
                            f"P{pa+1}:C{i+1} ↔ P{pb+1}:C{j+1} "
                            f"| similarity={score:.3f}"
                        )
                        found = True

            if not found:
                print(f"No pairs >= {threshold:.2f}")

## 8. One command interface

In [12]:
def inspect(what, threshold=0.70):
    options = {
        "text_boxes": lambda: show_text_boxes(per_page_results),
        "char_boxes": lambda: show_char_boxes(per_page_results),
        "panel_boxes": lambda: show_panel_boxes(per_page_results),
        "tail_boxes": lambda: show_tail_boxes(per_page_results),
        "text_char": lambda: show_text_char(per_page_results),
        "text_tail": lambda: show_text_tail(per_page_results),
        "same_char_page": lambda: show_same_char_page(per_page_results),
        "cross_page_candidates": lambda: show_cross_page_candidates(
            per_page_results,
            character_embeddings,
            threshold=threshold,
        ),
        "ocr": lambda: show_ocr(per_page_results),
    }

    if what == "all":
        for name, fn in options.items():
            print(f"\n\n######## {name.upper()} ########")
            fn()
        return

    if what not in options:
        print("Choose one of:")
        print(list(options.keys()) + ["all"])
        return

    options[what]()

## 9. Examples

In [17]:
# inspect("text_char")

# inspect("text_boxes")
# inspect("char_boxes")
# inspect("panel_boxes")
# inspect("tail_boxes")
# inspect("same_char_page")
inspect("cross_page_candidates", threshold=0.75)
# inspect("ocr")
# inspect("all")


===== PAGE 1 ↔ PAGE 2 =====
P1:C1 ↔ P2:C1 | similarity=0.770
P1:C1 ↔ P2:C7 | similarity=0.804
P1:C1 ↔ P2:C12 | similarity=0.923
P1:C2 ↔ P2:C2 | similarity=0.867
P1:C3 ↔ P2:C3 | similarity=0.755
P1:C3 ↔ P2:C12 | similarity=0.823
P1:C4 ↔ P2:C7 | similarity=0.799
P1:C4 ↔ P2:C8 | similarity=0.761
P1:C4 ↔ P2:C12 | similarity=0.940
P1:C5 ↔ P2:C1 | similarity=0.756
P1:C5 ↔ P2:C12 | similarity=0.944
P1:C6 ↔ P2:C1 | similarity=0.752
P1:C6 ↔ P2:C12 | similarity=0.947
P1:C8 ↔ P2:C6 | similarity=0.767
P1:C9 ↔ P2:C4 | similarity=0.958
P1:C9 ↔ P2:C8 | similarity=0.780
P1:C9 ↔ P2:C10 | similarity=0.973
P1:C9 ↔ P2:C11 | similarity=0.968
P1:C10 ↔ P2:C1 | similarity=0.852
P1:C10 ↔ P2:C4 | similarity=0.826
P1:C10 ↔ P2:C7 | similarity=0.803
P1:C10 ↔ P2:C10 | similarity=0.802
P1:C10 ↔ P2:C11 | similarity=0.776

===== PAGE 1 ↔ PAGE 3 =====
P1:C2 ↔ P3:C5 | similarity=0.873
P1:C3 ↔ P3:C3 | similarity=0.772
P1:C3 ↔ P3:C6 | similarity=0.835
P1:C8 ↔ P3:C1 | similarity=0.804
P1:C9 ↔ P3:C2 | similarity=0.942
P1:C

## 10. Raw sequence dictionary

In [14]:
def get_sequence_summary(results):
    out = []

    for page_idx, r in enumerate(results):
        out.append({
            "page": PAGE_FILES[page_idx].name,
            "text_boxes": r["texts"],
            "character_boxes": r["characters"],
            "panel_boxes": r["panels"],
            "tail_boxes": r["tails"],
            "text_character_associations": r["text_character_associations"],
            "text_tail_associations": r["text_tail_associations"],
            "character_cluster_labels": r["character_cluster_labels"],
            "is_essential_text": r["is_essential_text"],
            "ocr": r.get("ocr", []),
        })

    return out

sequence_summary = get_sequence_summary(per_page_results)
sequence_summary[0]

{'page': '01.png',
 'text_boxes': [[491.7005615234375,
   115.26335144042969,
   585.6625366210938,
   249.1288604736328],
  [284.1544494628906, 145.4463653564453, 410.4164733886719, 238.1248779296875],
  [117.97957611083984, 137.41587829589844, 231.5010528564453, 242.34912109375],
  [649.3109741210938, 615.7588500976562, 736.6842041015625, 685.5227661132812],
  [398.9065246582031, 592.5792846679688, 482.6698303222656, 712.1756591796875],
  [660.7257080078125, 899.6484985351562, 733.7633056640625, 961.0811157226562],
  [325.44903564453125, 628.831787109375, 358.965576171875, 648.5642700195312]],
 'character_boxes': [[514.8379516601562,
   904.9801635742188,
   722.2035522460938,
   1100.118408203125],
  [513.4190063476562, 945.6310424804688, 568.0812377929688, 1056.04833984375],
  [385.3322448730469, 861.139892578125, 528.3566284179688, 1100.47705078125],
  [376.87567138671875, 206.06451416015625, 577.2099609375, 559.404296875],
  [417.177001953125, 585.4215087890625, 705.3186645507812